# A3 - Daylight screening demonstration

This notebook demonstrates room-level results from the IFC analysis. It does not use a GUI. Set the path to a locally stored IFC model before running. The 10% window-to-floor-area check is a simplified screening method, **not a full daylight compliance assessment**.

In [14]:
from pathlib import Path
import sys

project_root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "A3" / "daylight_analysis.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from A3.daylight_analysis import run_ifc_analysis
from A3.daylight_tables import make_tables, export_csv
import pandas as pd

In [15]:
model_path = project_root / "model" / "B308X.ifc"
assert model_path.is_file(), f"Set model_path to an existing IFC file: {model_path}"
results = run_ifc_analysis(str(model_path))
rows, summary = make_tables(results)

## Room-level results

In [16]:
room_table = pd.DataFrame(rows)
display(room_table.head(20))

,Storey,Room,Use,Floor area (m²),Window area (m²),Windows,Window-to-floor ratio (%),Screening status
0,Temp. Level 1,2,Area,2800.654800,403.374153,96,14.402852,REVIEW USE
1,Temp. Level 1,3,Area,2647.174800,467.601658,76,17.664178,REVIEW USE
2,E2_2. sal,4,Area,137.437781,0.000000,0,0.000000,REVIEW USE


## Summary

In [17]:
display(pd.DataFrame([summary]).T.rename(columns={0:"Value"}))
display(room_table.groupby("Screening status", dropna=False).size().rename("Rooms").to_frame())

,Value
total_floor_area,5585.267381
total_window_area,870.975811
rooms_with_windows,2.000000
total_rooms,3.000000
total_windows,229.000000
windows_assigned,172.000000
overall_ratio,15.594165
OK,0.000000
FAIL,0.000000
REVIEW USE,3.000000


,Rooms
Screening status,
REVIEW USE,3


## Export

In [18]:
export_csv(rows, "results.csv")
print("Saved results.csv")

Saved results.csv


## Interpretation and limitations

The tool estimates window-to-floor-area ratios from IFC quantities and geometric window-to-space assignment. Room classification is based on name keywords and must be reviewed. The calculation does not account for daylight transmission, obstruction, shading, window orientation, or the full BR18 documentation conditions. Unassigned windows and missing floor areas require manual checking.